Install Dependencies

In [ ]:

!pip install sentence-transformers faiss-cpu pyarrow pandas xgboost shap tqdm scikit-learn numpy

import torch
if torch.cuda.is_available():
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'faiss-gpu'], capture_output=True)

print(' Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.4 MB/s eta 0:00:00
 Dependencies installed


Configuration

In [ ]:
# ── Path to your dataset on Google Drive ──────────────────────────
PARQUET_PATH = "/content/drive/MyDrive/training_data.parquet"   # ← change this

# Set True if PARQUET_PATH points to a folder of .parquet files
IS_PARTITIONED_DIR = False

# ── Paths where the built index + classifier are saved ────────────
# These are written to Drive so you never rebuild from scratch again
INDEX_SAVE_PATH    = "/content/drive/MyDrive/adr_faiss.index"
METADATA_SAVE_PATH = "/content/drive/MyDrive/adr_metadata.parquet"
MODEL_SAVE_PATH    = "/content/drive/MyDrive/adr_xgb_model.json"

# ── Column names (auto-detected, override here if wrong) ──────────
TEXT_COLUMN  = "description"   # Main text field to embed
LABEL_COLUMN = "final_label"   # Binary label: 1=suitable, 0=not suitable

# ── Embedding model ───────────────────────────────────────────────
# all-MiniLM-L6-v2 → fast, 384-dim, great quality/speed tradeoff
# For higher accuracy (slower): "BAAI/bge-small-en-v1.5"
EMBEDDING_MODEL    = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM      = 384
EMBEDDING_BATCH    = 512   # Reduce to 128 if you get OOM
MAX_SEQ_LENGTH     = 256   # Token limit per text

# ── FAISS settings ────────────────────────────────────────────────
# FAISS_NLIST: None = auto (4*sqrt(N)), or set manually e.g. 4096
FAISS_NLIST  = None
FAISS_M_PQ   = 96    # PQ sub-quantizers — must divide EMBEDDING_DIM
FAISS_NPROBE = 64    # Search-time probes: higher = more accurate, slower

TOP_K = 20   # Similar cases to retrieve per query

CHUNK_SIZE = 100_000

CLASSIFIER_SAMPLE_FRAC = 0.3
XGB_N_ESTIMATORS = 300
XGB_MAX_DEPTH    = 6
XGB_LEARNING_RATE = 0.05

print('Config loaded')

Mount Google Drive

In [ ]:
import os

IN_COLAB  = os.path.exists('/content')
IN_KAGGLE = os.path.exists('/kaggle')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✅ Google Drive mounted')
elif IN_KAGGLE:
    print('ℹ️  Kaggle detected — update PARQUET_PATH to /kaggle/input/.../file.parquet')
else:
    print('ℹ️  Local environment — update PARQUET_PATH to your file')

Imports & Helpers

In [ ]:
import numpy as np
import pandas as pd
import faiss
import torch
import gc
import math
import time
import warnings
import pickle
from pathlib import Path
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import xgboost as xgb
import shap
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_GPU_FAISS = DEVICE == 'cuda'
print(f'🖥️  Device     : {DEVICE}')
print(f'📦  FAISS ver  : {faiss.__version__}')
print(f'🤖  XGBoost    : {xgb.__version__}')

# ── Schema auto-detection ──────────────────────────────────────────
def detect_schema(columns):
    text_cands  = ['description','case_description','facts','plaint','text','content','judgment_text']
    label_cands = ['final_label','adr_label','odr_label','adr_target','odr_target','label']
    id_cands    = ['case_id','id','case_no','cnr_number']
    meta_cands  = ['case_id','court_level','court_name','year','state','case_type','act','section',
                   'title','disposal_nature','is_criminal','is_bailable','adr_label','odr_label',
                   'final_label','label_reason','llm_confidence','source']
    aux_cands   = ['title','case_type','act','section','disposal_nature','label_reason','state']
    return {
        'text'     : next((c for c in text_cands  if c in columns), columns[0]),
        'label'    : next((c for c in label_cands if c in columns), None),
        'case_id'  : next((c for c in id_cands    if c in columns), None),
        'metadata' : [c for c in meta_cands if c in columns],
        'aux_text' : [c for c in aux_cands  if c in columns],
    }

# ── Build rich embedding text ──────────────────────────────────────
def build_text(row, text_col, aux_cols):
    parts = [str(row.get(text_col) or '').strip()[:1500]]
    for col in aux_cols:
        v = str(row.get(col) or '').strip()
        if v and v.lower() not in ('nan','none','<na>','null',''):
            parts.append(f'{col}: {v}')
    return ' | '.join(p for p in parts if p)

def build_texts(df, text_col, aux_cols):
    return [build_text(row, text_col, aux_cols) for _, row in df.iterrows()]

# ── Parquet helpers ────────────────────────────────────────────────
def get_files(path, is_dir):
    p = Path(path)
    if is_dir:
        return sorted(str(f) for f in p.glob('**/*.parquet')) or sorted(str(f) for f in p.glob('*.parquet'))
    return [str(p)]

def chunked_reader(files, columns, chunk_size):
    buf, buf_sz = [], 0
    for fp in files:
        for batch in pq.ParquetFile(fp).iter_batches(batch_size=chunk_size, columns=columns):
            df = batch.to_pandas()
            buf.append(df); buf_sz += len(df)
            if buf_sz >= chunk_size:
                merged = pd.concat(buf, ignore_index=True)
                yield merged.iloc[:chunk_size]
                rest = merged.iloc[chunk_size:]
                buf, buf_sz = ([rest], len(rest)) if len(rest) else ([], 0)
    if buf:
        yield pd.concat(buf, ignore_index=True)

print('✅ Helpers loaded')

Inspect Dataset

In [ ]:
FILES = get_files(PARQUET_PATH, IS_PARTITIONED_DIR)
print(f'📂 {len(FILES)} parquet file(s)')

pf = pq.ParquetFile(FILES[0])
COLUMNS = [f.name for f in pf.schema_arrow]
TOTAL_ROWS = sum(pq.read_metadata(f).num_rows for f in FILES)

print(f'📊 Total rows : {TOTAL_ROWS:,}')
print(f'📐 Columns    : {COLUMNS}')

sample = pf.read_row_group(0).to_pandas().head(3)
print('\nSample:')
display(sample)

# Apply schema
SCHEMA = detect_schema(COLUMNS)
SCHEMA['text']     = TEXT_COLUMN  if TEXT_COLUMN  in COLUMNS else SCHEMA['text']
SCHEMA['label']    = LABEL_COLUMN if LABEL_COLUMN in COLUMNS else SCHEMA['label']
SCHEMA['aux_text'] = [c for c in SCHEMA['aux_text'] if c in COLUMNS]

# Auto nlist
if FAISS_NLIST is None:
    FAISS_NLIST = max(256, min(int(4 * math.sqrt(TOTAL_ROWS)), 65536))

# Label distribution
lbl_col = SCHEMA['label']
lbl_sample = sample[lbl_col].value_counts() if lbl_col in sample.columns else None

print(f'\n✅ Schema: text={SCHEMA["text"]}  label={SCHEMA["label"]}')
print(f'   aux_text : {SCHEMA["aux_text"]}')
print(f'   FAISS nlist: {FAISS_NLIST:,}')
if lbl_sample is not None:
    print(f'   Label dist (sample): {lbl_sample.to_dict()}')

Load Embedding Model

In [ ]:
print(f'Loading {EMBEDDING_MODEL} on {DEVICE}...')
EMBEDDER = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDER.max_seq_length = MAX_SEQ_LENGTH

# Verify dim
test_dim = EMBEDDER.encode(['test'], show_progress_bar=False).shape[1]
assert test_dim == EMBEDDING_DIM, f'Dim mismatch: model={test_dim}, config={EMBEDDING_DIM}. Set EMBEDDING_DIM={test_dim}'
print(f'✅ Embedding model loaded — output dim: {test_dim}')

Build FAISS Index

In [ ]:
def build_faiss_index():
    if Path(INDEX_SAVE_PATH).exists() and Path(METADATA_SAVE_PATH).exists():
        print('✅ Index already exists — skipping build.')
        print(f'   Delete {INDEX_SAVE_PATH} to force rebuild.')
        return

    text_col  = SCHEMA['text']
    aux_cols  = SCHEMA['aux_text']
    meta_cols = SCHEMA['metadata']
    label_col = SCHEMA['label']
    needed    = list(set([text_col, label_col] + aux_cols + meta_cols) & set(COLUMNS))

    # ── Phase 1: sample training vectors ──────────────────────────
    n_train    = min(max(10 * FAISS_NLIST, 100_000), TOTAL_ROWS)
    samp_rate  = n_train / TOTAL_ROWS
    print(f'Phase 1 — sampling {n_train:,} vectors for IVF training...')

    train_texts = []
    for chunk in chunked_reader(FILES, needed, CHUNK_SIZE):
        s = chunk.sample(n=min(int(len(chunk)*samp_rate)+1, len(chunk)), random_state=42)
        train_texts.extend(build_texts(s, text_col, aux_cols))
        if len(train_texts) >= n_train:
            break

    train_emb = EMBEDDER.encode(
        train_texts[:n_train], batch_size=EMBEDDING_BATCH,
        show_progress_bar=True, normalize_embeddings=True
    ).astype('float32')
    del train_texts; gc.collect()

    # ── Phase 2: train FAISS IVF+PQ ───────────────────────────────
    print(f'Phase 2 — training FAISS IVF+PQ (nlist={FAISS_NLIST}, m_pq={FAISS_M_PQ})...')
    quantizer = faiss.IndexFlatL2(EMBEDDING_DIM)
    index     = faiss.IndexIVFPQ(quantizer, EMBEDDING_DIM, FAISS_NLIST, FAISS_M_PQ, 8)

    if USE_GPU_FAISS:
        try:
            res   = faiss.StandardGpuResources()
            index = faiss.index_cpu_to_gpu(res, 0, index)
            print('  (training on GPU)')
        except Exception as e:
            print(f'  GPU unavailable ({e}), using CPU')

    index.train(train_emb)
    del train_emb; gc.collect()
    print('  Training done.')

    # ── Phase 3: add all vectors ───────────────────────────────────
    print(f'Phase 3 — adding {TOTAL_ROWS:,} vectors in chunks of {CHUNK_SIZE:,}...')
    all_meta = []
    t0 = time.time()

    with tqdm(total=TOTAL_ROWS, unit='rows') as pbar:
        for chunk in chunked_reader(FILES, needed, CHUNK_SIZE):
            embs = EMBEDDER.encode(
                build_texts(chunk, text_col, aux_cols),
                batch_size=EMBEDDING_BATCH, show_progress_bar=False,
                normalize_embeddings=True
            ).astype('float32')
            index.add(embs)

            keep = [c for c in meta_cols + [text_col] if c in chunk.columns]
            all_meta.append(chunk[keep].copy())
            pbar.update(len(chunk))
            del embs; gc.collect()

    print(f'  Done in {(time.time()-t0)/60:.1f} min')

    # ── Save ───────────────────────────────────────────────────────
    print('Saving index...')
    cpu_index = faiss.index_gpu_to_cpu(index) if USE_GPU_FAISS else index
    faiss.write_index(cpu_index, INDEX_SAVE_PATH)

    print('Saving metadata...')
    pd.concat(all_meta, ignore_index=True).to_parquet(METADATA_SAVE_PATH, index=False)
    print('✅ Index build complete!')


build_faiss_index()

Load Index for Inference

In [ ]:
print('Loading FAISS index...')
INDEX = faiss.read_index(INDEX_SAVE_PATH)
INDEX.nprobe = FAISS_NPROBE

if USE_GPU_FAISS:
    try:
        res   = faiss.StandardGpuResources()
        INDEX = faiss.index_cpu_to_gpu(res, 0, INDEX)
        print('✅ Index on GPU')
    except:
        print('ℹ️  Index on CPU')
else:
    print('✅ Index on CPU')

print('Loading metadata...')
META_DF = pd.read_parquet(METADATA_SAVE_PATH)

assert INDEX.ntotal == len(META_DF), (
    f'MISMATCH: index={INDEX.ntotal:,} vectors, metadata={len(META_DF):,} rows — rebuild index'
)
print(f'✅ Vectors : {INDEX.ntotal:,}')
print(f'✅ Metadata: {len(META_DF):,} rows')

Train XGBoost Classifier  *(run once, saved to Drive)*

In [ ]:
def retrieve(query_emb, k=TOP_K):
    """Raw FAISS search — returns distances + row indices."""
    D, I = INDEX.search(query_emb.astype('float32'), k)
    return D[0], I[0]


def build_features(distances, indices, meta_df, schema):
    """
    Build a fixed-length feature vector from retrieved neighbors:
    - Top-K similarity scores (distance-based)
    - Weighted label vote (label * similarity)
    - Fraction of neighbors that are suitable
    - Metadata aggregates (is_criminal rate, court_level mode, etc.)
    """
    label_col = schema['label']
    valid = indices >= 0
    idx   = indices[valid]
    dist  = distances[valid]
    sim   = 1.0 / (1.0 + dist)   # L2 distance → similarity

    rows = meta_df.iloc[idx]
    feats = {}

    # Similarity stats
    feats['sim_mean'] = float(sim.mean()) if len(sim) else 0.0
    feats['sim_max']  = float(sim.max())  if len(sim) else 0.0
    feats['sim_min']  = float(sim.min())  if len(sim) else 0.0
    feats['sim_std']  = float(sim.std())  if len(sim) else 0.0

    # Per-rank similarity (pad with 0 if fewer than TOP_K)
    for i in range(TOP_K):
        feats[f'sim_{i}'] = float(sim[i]) if i < len(sim) else 0.0

    # Label-weighted vote from neighbors
    if label_col and label_col in rows.columns:
        labels = rows[label_col].fillna(0).astype(float).values
        feats['label_mean']          = float(labels.mean()) if len(labels) else 0.0
        feats['label_weighted_mean'] = float((labels * sim).sum() / (sim.sum() + 1e-9))
        feats['label_top3_mean']     = float(labels[:3].mean()) if len(labels) >= 3 else feats['label_mean']
        feats['label_top5_mean']     = float(labels[:5].mean()) if len(labels) >= 5 else feats['label_mean']
        feats['n_suitable']          = int(labels.sum())
        feats['n_not_suitable']      = int((1 - labels).sum())
    else:
        for k_ in ['label_mean','label_weighted_mean','label_top3_mean','label_top5_mean','n_suitable','n_not_suitable']:
            feats[k_] = 0.0

    # Metadata aggregates
    if 'is_criminal' in rows.columns:
        feats['criminal_rate'] = float(rows['is_criminal'].fillna(0).astype(float).mean())
    if 'is_bailable' in rows.columns:
        feats['bailable_rate'] = float(rows['is_bailable'].fillna(0).astype(float).mean())
    if 'court_level' in rows.columns:
        cl_map = {'District Court':0,'High Court':1,'Supreme Court':2,'Tribunal':3}
        feats['court_level_mode'] = float(rows['court_level'].map(cl_map).fillna(-1).mode().iloc[0] if len(rows) else -1)
    if 'year' in rows.columns:
        feats['year_mean'] = float(rows['year'].fillna(0).astype(float).mean())

    return feats


def train_classifier():
    if Path(MODEL_SAVE_PATH).exists():
        print('✅ Classifier already exists — skipping training.')
        print(f'   Delete {MODEL_SAVE_PATH} to retrain.')
        return

    label_col = SCHEMA['label']
    text_col  = SCHEMA['text']
    aux_cols  = SCHEMA['aux_text']
    needed    = list(set([text_col, label_col] + SCHEMA['metadata']) & set(COLUMNS))

    n_train = max(1, int(TOTAL_ROWS * CLASSIFIER_SAMPLE_FRAC))
    print(f'Collecting {n_train:,} training samples ({CLASSIFIER_SAMPLE_FRAC*100:.0f}% of dataset)...')

    X_rows, y_vals = [], []
    collected = 0

    with tqdm(total=n_train, desc='Building features', unit='rows') as pbar:
        for chunk in chunked_reader(FILES, needed, CHUNK_SIZE):
            remaining = n_train - collected
            if remaining <= 0:
                break
            chunk = chunk.sample(n=min(remaining, len(chunk)), random_state=42)
            chunk = chunk.dropna(subset=[label_col])

            texts  = build_texts(chunk, text_col, aux_cols)
            embs   = EMBEDDER.encode(
                texts, batch_size=EMBEDDING_BATCH,
                show_progress_bar=False, normalize_embeddings=True
            ).astype('float32')

            for i, (_, row) in enumerate(chunk.iterrows()):
                D, I = retrieve(embs[i:i+1])
                feats = build_features(D, I, META_DF, SCHEMA)
                X_rows.append(feats)
                y_vals.append(int(row[label_col]))

            collected += len(chunk)
            pbar.update(len(chunk))
            del embs; gc.collect()

    print(f'\nBuilt {len(X_rows):,} feature vectors. Training XGBoost...')
    X = pd.DataFrame(X_rows).fillna(0)
    y = np.array(y_vals)

    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

    scale_pos = int((y_tr == 0).sum()) / max(1, int((y_tr == 1).sum()))

    clf = xgb.XGBClassifier(
    n_estimators      = XGB_N_ESTIMATORS,
    max_depth         = XGB_MAX_DEPTH,
    learning_rate     = XGB_LEARNING_RATE,
    objective         = 'multi:softprob',   # ← add this
    num_class         = 3,                  # ← add this
    tree_method       = 'gpu_hist' if DEVICE == 'cuda' else 'hist',
    eval_metric       = 'mlogloss',         # ← change from 'auc' to this
    early_stopping_rounds = 20,
    random_state      = 42,
    n_jobs            = -1,
)

    clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=50)

    # Evaluate
    y_pred  = clf.predict(X_val)
    y_prob  = clf.predict_proba(X_val)[:, 1]
    auc     = roc_auc_score(y_val, y_prob)

    print('\n── Validation Results ───────────────────────────────────')
    print(classification_report(y_val, y_pred, target_names=['Not Suitable','ADR/ODR Suitable']))
    print(f'ROC-AUC : {auc:.4f}')

    clf.save_model(MODEL_SAVE_PATH)
    # Save feature names for inference
    feat_path = MODEL_SAVE_PATH.replace('.json', '_features.pkl')
    with open(feat_path, 'wb') as f:
        pickle.dump(list(X.columns), f)

    print(f'\n✅ Model saved to {MODEL_SAVE_PATH}')
    print(f'✅ Feature list saved to {feat_path}')


train_classifier()

Load Classifier

In [ ]:
CLF = xgb.XGBClassifier()
CLF.load_model(MODEL_SAVE_PATH)

feat_path = MODEL_SAVE_PATH.replace('.json', '_features.pkl')
with open(feat_path, 'rb') as f:
    FEATURE_NAMES = pickle.load(f)

# Pre-build SHAP explainer
EXPLAINER = shap.TreeExplainer(CLF)

print(f'✅ Classifier loaded — {len(FEATURE_NAMES)} features')

Predict ADR/ODR Suitability


In [ ]:
# ─── PASTE YOUR CASE DESCRIPTION HERE ────────────────────────────
QUERY_CASE = """
Petitioner M/s ABC Trading Co. filed a suit against M/s XYZ Exports Pvt. Ltd.
for recovery of Rs. 45 lakhs arising from non-payment of goods supplied under
a commercial contract dated 15 March 2022. Respondent denies liability claiming
the goods were defective. Both are private commercial entities. The contract
contains an arbitration clause at Section 12.
""".strip()
# ─────────────────────────────────────────────────────────────────

def predict(query_text):
    t0 = time.time()

    # 1. Embed query
    q_emb = EMBEDDER.encode(
        [query_text], normalize_embeddings=True, show_progress_bar=False
    ).astype('float32')

    # 2. Retrieve neighbors
    D, I = retrieve(q_emb)
    t_ret = time.time() - t0

    # 3. Build features
    feats = build_features(D, I, META_DF, SCHEMA)
    X     = pd.DataFrame([feats]).reindex(columns=FEATURE_NAMES, fill_value=0)

    # 4. Predict
    label = int(CLF.predict(X)[0])
    prob  = float(CLF.predict_proba(X)[0][1])  # P(suitable)
    t_inf = time.time() - t0

    # 5. SHAP explanation
    sv       = EXPLAINER.shap_values(X)
    shap_df  = pd.DataFrame({'feature': FEATURE_NAMES, 'shap': sv[0]})
    shap_df  = shap_df.reindex(shap_df['shap'].abs().sort_values(ascending=False).index)

    # 6. Retrieved cases summary
    valid    = I[I >= 0]
    sim      = 1.0 / (1.0 + D[I >= 0])
    ret_rows = META_DF.iloc[valid].copy()
    ret_rows['similarity'] = sim

    return {
        'label'          : label,
        'verdict'        : 'ADR/ODR SUITABLE' if label == 1 else 'NOT SUITABLE FOR ADR/ODR',
        'probability'    : prob,
        'confidence'     : 'High' if abs(prob-0.5) > 0.3 else 'Medium' if abs(prob-0.5) > 0.15 else 'Low',
        'shap_df'        : shap_df,
        'retrieved_cases': ret_rows,
        'retrieval_s'    : round(t_ret, 3),
        'total_s'        : round(t_inf, 3),
    }


result = predict(QUERY_CASE)

# ── Print verdict ─────────────────────────────────────────────────
print('\n' + '='*58)
print('  ADR/ODR SUITABILITY VERDICT')
print('='*58)
verdict_icon = '✅' if result['label'] == 1 else '❌'
print(f"{verdict_icon}  Verdict      : {result['verdict']}")
print(f"   Probability  : {result['probability']:.1%}  (P=suitable)")
print(f"   Confidence   : {result['confidence']}")
print(f"   Timing       : retrieve={result['retrieval_s']}s  total={result['total_s']}s")

print('\n── Top Feature Drivers (SHAP) ───────────────────────────')
top_shap = result['shap_df'].head(8)
for _, r in top_shap.iterrows():
    bar   = '█' * int(abs(r['shap']) * 60)
    sign  = '+' if r['shap'] > 0 else '-'
    print(f"  {sign} {r['feature']:<28} {r['shap']:+.4f}  {bar}")

print('\n── Neighbor Label Distribution ─────────────────────────')
lbl_col = SCHEMA['label']
if lbl_col in result['retrieved_cases'].columns:
    vc = result['retrieved_cases'][lbl_col].value_counts().to_dict()
    print(f"  Suitable (1)     : {vc.get(1, 0):>4}")
    print(f"  Not Suitable (0) : {vc.get(0, 0):>4}")
print('='*58)

View Retrieved Similar Cases

In [ ]:
ret = result['retrieved_cases']
lbl_col = SCHEMA['label']

show_cols = ['similarity'] + [
    c for c in ['case_id','court_name','court_level','year','state',
                 'case_type','disposal_nature','is_criminal', lbl_col,'label_reason']
    if c in ret.columns
]

display(
    ret[show_cols].head(TOP_K)
    .style
    .background_gradient(subset=['similarity'], cmap='Blues')
    .format({'similarity': '{:.4f}'})
)

SHAP Visualisation

In [ ]:
import matplotlib.pyplot as plt

lbl_col = SCHEMA['label']
q_emb = EMBEDDER.encode([QUERY_CASE], normalize_embeddings=True, show_progress_bar=False).astype('float32')
D, I  = retrieve(q_emb)
feats = build_features(D, I, META_DF, SCHEMA)
X_q   = pd.DataFrame([feats]).reindex(columns=FEATURE_NAMES, fill_value=0)
sv    = EXPLAINER.shap_values(X_q)

# Waterfall-style bar chart
shap_series = pd.Series(sv[0], index=FEATURE_NAMES).sort_values(key=abs, ascending=False).head(12)
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in shap_series.values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(shap_series.index[::-1], shap_series.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('SHAP value  (→ pushes toward Suitable,  ← pushes toward Not Suitable)')
ax.set_title(f'Why this prediction?   Verdict: {result["verdict"]}   P={result["probability"]:.1%}')
plt.tight_layout()
plt.show()

Batch Prediction

In [ ]:
def batch_predict(input_path, output_path, text_col=None, max_rows=None):
    text_col = text_col or SCHEMA['text']
    df = pd.read_parquet(input_path) if input_path.endswith('.parquet') else pd.read_csv(input_path)
    if max_rows:
        df = df.head(max_rows)

    print(f'Batch predicting {len(df):,} cases...')
    preds = []

    # Embed all at once in batches (much faster than one-by-one)
    texts = df[text_col].fillna('').astype(str).tolist()
    all_embs = EMBEDDER.encode(
        texts, batch_size=EMBEDDING_BATCH,
        show_progress_bar=True, normalize_embeddings=True
    ).astype('float32')

    feat_list = []
    for i in tqdm(range(len(texts)), desc='Retrieving'):
        D, I   = retrieve(all_embs[i:i+1])
        feats  = build_features(D, I, META_DF, SCHEMA)
        feat_list.append(feats)

    X_all = pd.DataFrame(feat_list).reindex(columns=FEATURE_NAMES, fill_value=0)
    labels = CLF.predict(X_all)
    probs  = CLF.predict_proba(X_all)[:, 1]

    df['predicted_label']       = labels
    df['predicted_probability'] = probs.round(4)
    df['predicted_verdict']     = df['predicted_label'].map({1: 'ADR/ODR SUITABLE', 0: 'NOT SUITABLE'})

    df.to_parquet(output_path, index=False)
    print(f'\n✅ Saved to {output_path}')
    print(f'   Suitable     : {(labels==1).sum():,}')
    print(f'   Not Suitable : {(labels==0).sum():,}')
    display(df[['predicted_label','predicted_probability','predicted_verdict']].head(5))
    return df


# ── Uncomment to run ──────────────────────────────────────────────
# df_out = batch_predict(
#     input_path  = "/content/drive/MyDrive/new_cases.csv",
#     output_path = "/content/drive/MyDrive/new_cases_predicted.parquet",
#     max_rows    = 500
# )

print('✅ Batch prediction ready — uncomment above to use')